First we sanity check the corpus...
Let's look at the stories generated and visually ensure they make sense.

In [2]:
import random
import textwrap
import sys

from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

CORPUS_DIR = PROJECT_ROOT / "datasets" / "qwen-emotion-stories" / "corpus"

from core.shards import read_shards
from core.utils import emotion_words_named

/Users/folusoogunlana/code/oss/emotion-concepts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from collections import Counter

rows = read_shards(CORPUS_DIR, sample=False)
print(f"{len(rows)} stories")
print(Counter(r["emotion"] for r in rows))

807 stories
Counter({'neutral': 98, 'surprised': 91, 'ashamed': 68, 'disgusted': 65, 'desperate': 64, 'afraid': 59, 'sad': 59, 'calm': 58, 'angry': 54, 'proud': 52, 'excited': 50, 'joyful': 48, 'lonely': 41})


In [4]:
def show_samples(n: int = 2, seed: int | None = None, emotion: str | None = None, sample: bool = True, prompts: bool = True) -> None:
    rows = read_shards(CORPUS_DIR, sample=sample)
    if emotion:
        rows = [r for r in rows if r["emotion"] == emotion]

    for row in random.Random(seed).sample(rows, min(n, len(rows))):
        named = emotion_words_named(row["text"])
        print("=" * 88)
        print(f"{row['emotion']}  |  {row['topic']}  |  #{row['index']}")
        print(f"{len(row['text'].split())} words  |  ends: {row['text'].rstrip()[-1]!r}  "
              f"|  names: {named or 'none'}")
        if prompts:
            print("-" * 88)
            print(textwrap.indent(row["prompt"], "  "))
        print("-" * 88)
        print(textwrap.fill(row["text"], width=88))
        print()

show_samples(2, sample=False)

angry  |  Someone discovers their mother kept every school assignment  |  #1
100 words  |  ends: '.'  |  names: none
----------------------------------------------------------------------------------------
  Write a short story (roughly one paragraph) based on the following premise.

  Topic: Someone discovers their mother kept every school assignment

  The story should follow a character who is feeling angry.

  Write the story in English. Use either third-person or first-person narration.

  Write between 90 and 130 words. Finish the final sentence. Do not write a title.

  The character is ALREADY feeling angry in the very first sentence. Open inside the scene, at the moment the feeling is strongest. Do not begin with backstory, scene-setting, or a build-up towards the feeling, and do not begin with "Once upon a time".

  ONE STATE ONLY: the character feels angry and nothing else, from the first word to the last. Nothing in the story relieves, resolves, or complicates the feeling -

Next up we need to extract activations from all layers to do our difference of means probe

In [5]:
## get activations

from core.models import Model

m = Model()
m.load_weights()

Loading weights: 100%|██████████| 290/290 [00:01<00:00, 172.14it/s]


In [6]:
import torch

# Extract activations
# Create a function for extracting activations
# The function should take in a model and a list of texts, and it should return a list of tensors of activations
# For each prompt, there should be an activation with the shape of the residual stream
# Split the texts into train and test before passing into the function - only pass train
# TODO: skip until 20 or 18
def extract_activations(texts: list[str], batch_size: int = 16, layers: list[int] = None):
    # iterate over texts (they will be specific to an emotion)
    # for each text, teacher force into model and get hidden states
    # get hidden states in each layer and mean-pool them
    # return activations
    m.tok.padding_side = "right"
    final_states = {}

    if layers is None:
        layers = list(range(m.model.config.num_hidden_layers + 1))
    for b in range(0, len(texts), batch_size):
        inputs = m.tok(texts[b : b + batch_size], padding=True, return_tensors='pt', truncation=False).to(m.device)

        with torch.no_grad():
            outputs = m.model(input_ids=inputs.input_ids, output_hidden_states=True)
        
        mask = inputs.attention_mask.unsqueeze(-1)
        # shape: b, seq, 1

        for l in layers:
            hidden_states = (outputs.hidden_states[l] * mask).sum(1) / mask.sum(1)
            # shape: b, d_model

            if final_states.get(l):
                final_states[l] = torch.cat([final_states[l], hidden_states], dim=0)
            else:
                final_states[l] = hidden_states
    return final_states

activations = extract_activations([r["text"] for r in rows[:16]])


In [11]:
# Emotion vectors
# Create a function for creating emotion vectors
# It should take an emotion and all the texts, and it should spit out the emotion vector
# It should take the mean of all the train activations for each emotion - e_mean
# It should take the mean of all activations completely - mean
# It should subtract e_mean - mean to find the emotion vector

def emotion_vectors(texts: list[str], batch_size: int = 16):
    # for each emotion (apart from neutral)
        # get all the train texts for the specific emotion
        # get activations for those texts
        # store mean activations for emotion
    # subtract out mean of all emotions
    # return list of emotion vectors
    counts = Counter(r["emotion"] for r in rows)
    emotions = sorted(e for e in counts if e != "neutral")
    vectors = {}

    train = [t for t in texts if t["split"] == "train"]
    acts = extract_activations([t["text"] for t in train], batch_size)
    global_mean = {}

    for l in range(len(acts)):
        global_mean[l] = acts[l].mean(0)

    for e in emotions:
        train_texts = [t for t in train if t["emotion"] == e]
        train_acts = extract_activations([t["text"] for t in train_texts])
        for l in range(len(train_acts)):
            pooled = train_acts[l].mean(0)
            # shape: d_model
            vectors[(e, l)] = pooled - global_mean[l]

    return vectors        

emovecs = emotion_vectors(rows[:16])

In [ ]:
# Denoising
# Create a function to denoise the emotion vectors in line with the paper
# To denoise do PCA on the set of the neutral activations
# Then after that, take the top components explaining 50% of variance and project them out from the emotion vectors
# Fit with PCA once per layer
# How do I confirm they have been denoised?? - 
    # Test with pairwise cosine similarity before and after. Also test with nearest centroid accuracy on held-out topics before vs after

def denoise(vecs: torch.Tensor, neutrals: list[str], batch_size: int = 16, frac = 0.5):
    # teacher force the neutrals to get their activations
    # do PCA on the neutral activations 
    # measure variance using eigenvalues as weights
    # take the first n vectors until 0.5 of eigenvalue density
    # project the neutral activations onto those n vectors (to get the right magnitude of noise)
    # do the above with a matrix multiplication
    # subtract out the projected neutral activations from the emotion vectors
    # return the emotion vectors
    counts = Counter(r["emotion"] for r in rows)
    emotions = sorted(e for e in counts if e != "neutral")
    acts = extract_activations(neutrals, batch_size)
    denoised = {}
    for l in range(len(acts)):
        X = acts[l].float()
        Xc = X - X.mean(0)
        _, S, Vh = torch.linalg.svd(Xc, full_matrice=False)
        ratio = (S ** 2) / (S ** 2).sum()
        k = int((ratio.cumsum() < frac).sum().item()) + 1
        pcs = Vh[:k]

        for e in emotions:
            v = vecs[(e, l)]
            denoised[l] = v - ((v @ pcs.T) @ pcs)
        
    return denoised

denoise(emovecs, [r["text"] for r in rows if r["emotion"] == "neutral"])

{}